In [1]:
# ============================================================
# WEEK 6 — VALIDATION AND RESEARCH CLAIM AUDIT
# Cell 1: Environment Setup and Imports
# ============================================================

import os
import sys
import warnings
import random

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
    TimeSeriesSplit
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("=" * 70)
print("WEEK 6 — VALIDATION AND RESEARCH CLAIM AUDIT")
print("=" * 70)

print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

print("\nEnvironment setup completed successfully.")

WEEK 6 — VALIDATION AND RESEARCH CLAIM AUDIT
Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
NumPy version: 2.1.3
Pandas version: 2.2.3

Environment setup completed successfully.


In [3]:
# ============================================================
# CELL 2 — CHECK CURRENT ENVIRONMENT
# ============================================================

import os

print("=" * 70)
print("CHECKING AVAILABLE DATA")
print("=" * 70)

print("\nCurrent working directory:")
print(os.getcwd())

print("\nFiles/folders available here:")
for item in os.listdir("."):
    print(" -", item)

CHECKING AVAILABLE DATA

Current working directory:
/content

Files/folders available here:
 - .config
 - sample_data


In [5]:
# ============================================================
# CELL 3 — DOWNLOAD KAGGLE HOUSING DATASET
# ============================================================

!pip -q install kagglehub

import kagglehub
import os

print("=" * 70)
print("DOWNLOADING KAGGLE HOUSING DATASET")
print("=" * 70)

# Download the public Kaggle dataset
dataset_path = kagglehub.dataset_download(
    "dansbecker/home-data-for-ml-course"
)

print("\nDataset downloaded successfully!")
print("Dataset path:")
print(dataset_path)

print("\nFiles found:")
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(" -", os.path.join(root, file))

DOWNLOADING KAGGLE HOUSING DATASET


100%|██████████| 94.0k/94.0k [00:00<00:00, 777kB/s]

Extracting files...

Dataset downloaded successfully!
Dataset path:
/root/.cache/kagglehub/datasets/dansbecker/home-data-for-ml-course/versions/1

Files found:
 - /root/.cache/kagglehub/datasets/dansbecker/home-data-for-ml-course/versions/1/train.csv


In [6]:
# ============================================================
# CELL 4 — LOAD AND INSPECT DATASET
# ============================================================

DATA_PATH = os.path.join(dataset_path, "train.csv")

# Load dataset
df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 70)

print(f"\nDataset shape: {df.shape}")

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

display(missing_values)

print("\nTarget column check:")
if "SalePrice" in df.columns:
    print("SalePrice found successfully.")
    print(f"SalePrice mean: {df['SalePrice'].mean():,.2f}")
    print(f"SalePrice median: {df['SalePrice'].median():,.2f}")
else:
    print("WARNING: SalePrice column was not found.")

DATASET LOADED SUCCESSFULLY

Dataset shape: (1460, 81)

First 5 rows:


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000



Column names:
['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'Pool

,0
Id,int64
MSSubClass,int64
MSZoning,object
LotFrontage,float64
LotArea,int64
...,...
MoSold,int64
YrSold,int64
SaleType,object
SaleCondition,object



Missing values:


,0
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
FireplaceQu,690
LotFrontage,259
GarageType,81
GarageYrBlt,81
GarageFinish,81



Target column check:
SalePrice found successfully.
SalePrice mean: 180,921.20
SalePrice median: 163,000.00


In [7]:
# ============================================================
# CELL 5 — TIME / CHRONOLOGY AUDIT
# ============================================================

print("=" * 70)
print("TIME / CHRONOLOGY AUDIT")
print("=" * 70)

# Check year distribution
print("\n1. Properties by Year Sold:")
year_counts = df["YrSold"].value_counts().sort_index()
display(year_counts.to_frame(name="Number_of_Sales"))

# Check month distribution
print("\n2. Properties by Month Sold:")
month_counts = df["MoSold"].value_counts().sort_index()
display(month_counts.to_frame(name="Number_of_Sales"))

# Create a chronological time variable
df["SalePeriod"] = (
    df["YrSold"].astype(str) + "-" +
    df["MoSold"].astype(str).str.zfill(2)
)

print("\n3. Earliest sale period:")
print(df["SalePeriod"].min())

print("\n4. Latest sale period:")
print(df["SalePeriod"].max())

# Check chronological ordering
print("\n5. First 10 records by sale year/month:")
display(
    df[["Id", "YrSold", "MoSold", "SalePeriod", "SalePrice"]]
    .sort_values(["YrSold", "MoSold"])
    .head(10)
)

print("\nTime audit completed.")

TIME / CHRONOLOGY AUDIT

1. Properties by Year Sold:


,Number_of_Sales
YrSold,
2006,314
2007,329
2008,304
2009,338
2010,175



2. Properties by Month Sold:


,Number_of_Sales
MoSold,
1,58
2,52
3,106
4,141
5,204
6,253
7,234
8,122
9,63



3. Earliest sale period:
2006-01

4. Latest sale period:
2010-07

5. First 10 records by sale year/month:


,Id,YrSold,MoSold,SalePeriod,SalePrice
141,142,2006,1,2006-01,260000
169,170,2006,1,2006-01,228000
302,303,2006,1,2006-01,205000
370,371,2006,1,2006-01,172400
411,412,2006,1,2006-01,145000
664,665,2006,1,2006-01,423000
810,811,2006,1,2006-01,181000
996,997,2006,1,2006-01,136500
1040,1041,2006,1,2006-01,155000
1404,1405,2006,1,2006-01,105000



Time audit completed.


In [8]:
# ============================================================
# CELL 6 — BASELINE RANDOM SPLIT
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=" * 70)
print("BASELINE MODEL — RANDOM TRAIN/TEST SPLIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Separate target from features
# ------------------------------------------------------------

X = df.drop(columns=["SalePrice", "SalePeriod"])
y = df["SalePrice"]

# Remove Id because it is only an identifier
X = X.drop(columns=["Id"])

# ------------------------------------------------------------
# 2. Identify numerical and categorical columns
# ------------------------------------------------------------

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"\nNumerical features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

# ------------------------------------------------------------
# 3. Preprocessing
# ------------------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# ------------------------------------------------------------
# 4. Random train/test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_SEED
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

# ------------------------------------------------------------
# 5. Random Forest model
# ------------------------------------------------------------

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]
)

# Train
baseline_model.fit(X_train, y_train)

# Predict
y_pred = baseline_model.predict(X_test)

# ------------------------------------------------------------
# 6. Evaluation
# ------------------------------------------------------------

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 70)
print("BASELINE RANDOM-SPLIT RESULTS")
print("=" * 70)

print(f"MAE : ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²  : {r2:.4f}")

BASELINE MODEL — RANDOM TRAIN/TEST SPLIT

Numerical features: 36
Categorical features: 43

Training samples: 1168
Testing samples:  292

BASELINE RANDOM-SPLIT RESULTS
MAE : $17,386.11
RMSE: $28,495.23
R²  : 0.8941


In [9]:
# ============================================================
# CELL 7 — HONEST TIME-AWARE SPLIT
# ============================================================

print("=" * 70)
print("HONEST TIME-AWARE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Sort the original data chronologically
# ------------------------------------------------------------

time_df = df.sort_values(
    ["YrSold", "MoSold"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Create a chronological cutoff
# ------------------------------------------------------------
# We use the final 20% of observations as the future test set.

split_index = int(len(time_df) * 0.80)

train_time = time_df.iloc[:split_index].copy()
test_time = time_df.iloc[split_index:].copy()

print(f"\nTotal observations : {len(time_df)}")
print(f"Training observations: {len(train_time)}")
print(f"Future test observations: {len(test_time)}")

print("\nTraining period:")
print(
    train_time["YrSold"].min(),
    "to",
    train_time["YrSold"].max(),
    "(last month:",
    train_time.iloc[-1]["MoSold"],
    ")"
)

print("\nFuture test period:")
print(
    test_time["YrSold"].min(),
    "to",
    test_time["YrSold"].max(),
    "(first month:",
    test_time.iloc[0]["MoSold"],
    ")"
)

# ------------------------------------------------------------
# 3. Prepare features and target
# ------------------------------------------------------------

X_time_train = train_time.drop(
    columns=["SalePrice", "SalePeriod"]
)

X_time_test = test_time.drop(
    columns=["SalePrice", "SalePeriod"]
)

y_time_train = train_time["SalePrice"]
y_time_test = test_time["SalePrice"]

# Remove identifier
X_time_train = X_time_train.drop(columns=["Id"])
X_time_test = X_time_test.drop(columns=["Id"])

# ------------------------------------------------------------
# 4. Build a fresh preprocessing pipeline
# ------------------------------------------------------------

numeric_features_time = X_time_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_time = X_time_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_transformer_time = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer_time = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_time = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_time, numeric_features_time),
        ("cat", categorical_transformer_time, categorical_features_time)
    ]
)

# ------------------------------------------------------------
# 5. Train Random Forest using only historical data
# ------------------------------------------------------------

time_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_time),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]
)

time_model.fit(X_time_train, y_time_train)

# ------------------------------------------------------------
# 6. Predict future observations
# ------------------------------------------------------------

y_time_pred = time_model.predict(X_time_test)

# ------------------------------------------------------------
# 7. Evaluate
# ------------------------------------------------------------

time_mae = mean_absolute_error(
    y_time_test,
    y_time_pred
)

time_rmse = np.sqrt(
    mean_squared_error(
        y_time_test,
        y_time_pred
    )
)

time_r2 = r2_score(
    y_time_test,
    y_time_pred
)

print("\n" + "=" * 70)
print("HONEST TIME-AWARE RESULTS")
print("=" * 70)

print(f"MAE : ${time_mae:,.2f}")
print(f"RMSE: ${time_rmse:,.2f}")
print(f"R²  : {time_r2:.4f}")

HONEST TIME-AWARE VALIDATION

Total observations : 1460
Training observations: 1168
Future test observations: 292

Training period:
2006 to 2009 (last month: 7 )

Future test period:
2009 to 2010 (first month: 7 )

HONEST TIME-AWARE RESULTS
MAE : $17,181.08
RMSE: $25,948.76
R²  : 0.8835


In [10]:
# ============================================================
# CELL 8 — BEFORE vs AFTER VALIDATION COMPARISON
# ============================================================

print("=" * 70)
print("BEFORE vs AFTER — VALIDATION COMPARISON")
print("=" * 70)

# Create comparison table
comparison_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Random_Split_Baseline": [
        mae,
        rmse,
        r2
    ],
    "Time_Aware_Split": [
        time_mae,
        time_rmse,
        time_r2
    ]
})

# Calculate percentage change
comparison_df["Percentage_Change"] = (
    (comparison_df["Time_Aware_Split"] -
     comparison_df["Random_Split_Baseline"])
    / comparison_df["Random_Split_Baseline"]
) * 100

display(comparison_df)

print("\nInterpretation:")

print(
    f"- MAE changed by "
    f"{comparison_df.loc[0, 'Percentage_Change']:.2f}%"
)

print(
    f"- RMSE changed by "
    f"{comparison_df.loc[1, 'Percentage_Change']:.2f}%"
)

print(
    f"- R² changed by "
    f"{comparison_df.loc[2, 'Percentage_Change']:.2f}%"
)

print("\nMethodology note:")
print(
    "The random split mixes observations across the dataset, "
    "while the time-aware split evaluates later observations "
    "using earlier observations for training."
)

BEFORE vs AFTER — VALIDATION COMPARISON


,Metric,Random_Split_Baseline,Time_Aware_Split,Percentage_Change
0,MAE,17386.105702,17181.075651,-1.179275
1,RMSE,28495.231222,25948.763666,-8.936469
2,R²,0.894140,0.883502,-1.189829



Interpretation:
- MAE changed by -1.18%
- RMSE changed by -8.94%
- R² changed by -1.19%

Methodology note:
The random split mixes observations across the dataset, while the time-aware split evaluates later observations using earlier observations for training.


In [11]:
# ============================================================
# CELL 9 — LEAKAGE AUDIT: TARGET & IDENTIFIER CHECK
# ============================================================

print("=" * 70)
print("LEAKAGE AUDIT — TARGET AND IDENTIFIER CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check whether target appears in feature columns
# ------------------------------------------------------------

feature_columns = [
    col for col in df.columns
    if col not in ["SalePrice", "SalePeriod"]
]

print("\n1. Target leakage check:")
print("Is 'SalePrice' present in features?",
      "SalePrice" in feature_columns)

# ------------------------------------------------------------
# 2. Check identifier
# ------------------------------------------------------------

print("\n2. Identifier check:")
print("Is 'Id' present in raw feature columns?",
      "Id" in feature_columns)

# ------------------------------------------------------------
# 3. Check exact duplicate columns
# ------------------------------------------------------------

print("\n3. Exact duplicate-column check:")

duplicate_columns = []

columns_to_check = [
    col for col in df.columns
    if col not in ["SalePrice", "SalePeriod"]
]

for i, col1 in enumerate(columns_to_check):
    for col2 in columns_to_check[i + 1:]:
        if df[col1].equals(df[col2]):
            duplicate_columns.append((col1, col2))

if duplicate_columns:
    print("Potential duplicate columns found:")
    for pair in duplicate_columns:
        print(" -", pair)
else:
    print("No exact duplicate feature columns found.")

# ------------------------------------------------------------
# 4. Numeric correlation with target
# ------------------------------------------------------------

print("\n4. Highest numeric correlations with SalePrice:")

numeric_df = df.select_dtypes(
    include=["int64", "float64"]
)

correlations = (
    numeric_df.corr()["SalePrice"]
    .drop("SalePrice")
    .abs()
    .sort_values(ascending=False)
)

display(correlations.head(15).to_frame(
    name="Absolute_Correlation_With_SalePrice"
))

print("\nLeakage audit step completed.")

LEAKAGE AUDIT — TARGET AND IDENTIFIER CHECK

1. Target leakage check:
Is 'SalePrice' present in features? False

2. Identifier check:
Is 'Id' present in raw feature columns? True

3. Exact duplicate-column check:
No exact duplicate feature columns found.

4. Highest numeric correlations with SalePrice:


,Absolute_Correlation_With_SalePrice
OverallQual,0.790982
GrLivArea,0.708624
GarageCars,0.640409
GarageArea,0.623431
TotalBsmtSF,0.613581
1stFlrSF,0.605852
FullBath,0.560664
TotRmsAbvGrd,0.533723
YearBuilt,0.522897
YearRemodAdd,0.507101



Leakage audit step completed.


In [12]:
# ============================================================
# CELL 10 — FUTURE INFORMATION / AVAILABILITY AUDIT
# ============================================================

print("=" * 70)
print("LEAKAGE AUDIT — FUTURE INFORMATION & FEATURE AVAILABILITY")
print("=" * 70)

# ------------------------------------------------------------
# Features related to the sale event
# ------------------------------------------------------------

sale_related_features = [
    "MoSold",
    "YrSold",
    "SaleType",
    "SaleCondition"
]

print("\n1. Sale-related features present in dataset:")

for feature in sale_related_features:
    if feature in df.columns:
        print(f" - {feature}")

# ------------------------------------------------------------
# Check whether any feature contains SalePrice directly
# ------------------------------------------------------------

print("\n2. Direct target-reference check:")

target_related_columns = [
    col for col in df.columns
    if "price" in col.lower()
]

print("Columns containing 'price' in their name:")
print(target_related_columns)

# ------------------------------------------------------------
# Inspect sale timing variables
# ------------------------------------------------------------

print("\n3. Sale timing variables:")

display(
    df[["YrSold", "MoSold", "SaleType", "SaleCondition"]].head(10)
)

# ------------------------------------------------------------
# Methodology assessment
# ------------------------------------------------------------

print("\n4. Availability assessment:")

print("""
- SalePrice is the prediction target and is excluded from model features.
- YrSold and MoSold describe when the property was sold.
- SaleType and SaleCondition describe the transaction/sale context.
- These variables require careful interpretation if the intended
  prediction task is supposed to happen BEFORE the sale event.
- They are not automatically leakage, but their availability depends
  on the exact prediction point being defined.
""")

print("\n5. Leakage conclusion:")

print(
    "No direct target leakage was detected. "
    "However, sale-event variables should be treated as "
    "potential availability risks depending on the prediction timing."
)

print("\nFuture-information audit completed.")

LEAKAGE AUDIT — FUTURE INFORMATION & FEATURE AVAILABILITY

1. Sale-related features present in dataset:
 - MoSold
 - YrSold
 - SaleType
 - SaleCondition

2. Direct target-reference check:
Columns containing 'price' in their name:
['SalePrice']

3. Sale timing variables:


,YrSold,MoSold,SaleType,SaleCondition
0,2008,2,WD,Normal
1,2007,5,WD,Normal
2,2008,9,WD,Normal
3,2006,2,WD,Abnorml
4,2008,12,WD,Normal
5,2009,10,WD,Normal
6,2007,8,WD,Normal
7,2009,11,WD,Normal
8,2008,4,WD,Abnorml
9,2008,1,WD,Normal



4. Availability assessment:

- SalePrice is the prediction target and is excluded from model features.
- YrSold and MoSold describe when the property was sold.
- SaleType and SaleCondition describe the transaction/sale context.
- These variables require careful interpretation if the intended
  prediction task is supposed to happen BEFORE the sale event.
- They are not automatically leakage, but their availability depends
  on the exact prediction point being defined.


5. Leakage conclusion:
No direct target leakage was detected. However, sale-event variables should be treated as potential availability risks depending on the prediction timing.

Future-information audit completed.


In [13]:
# ============================================================
# CELL 11 — REAL FAILURE EXAMPLES
# ============================================================

print("=" * 70)
print("REAL FAILURE EXAMPLES — BASELINE MODEL")
print("=" * 70)

# ------------------------------------------------------------
# 1. Create a table containing actual and predicted prices
# ------------------------------------------------------------

failure_df = X_test.copy()

failure_df["Actual_SalePrice"] = y_test.values
failure_df["Predicted_SalePrice"] = y_pred

# Absolute prediction error
failure_df["Absolute_Error"] = (
    failure_df["Actual_SalePrice"] -
    failure_df["Predicted_SalePrice"]
).abs()

# Percentage error
failure_df["Percentage_Error"] = (
    failure_df["Absolute_Error"] /
    failure_df["Actual_SalePrice"]
) * 100

# ------------------------------------------------------------
# 2. Show the worst absolute errors
# ------------------------------------------------------------

worst_absolute = failure_df.sort_values(
    "Absolute_Error",
    ascending=False
).head(10)

print("\nTOP 10 WORST PREDICTIONS BY ABSOLUTE ERROR:")

display(
    worst_absolute[
        [
            "Actual_SalePrice",
            "Predicted_SalePrice",
            "Absolute_Error",
            "Percentage_Error"
        ]
    ]
)

# ------------------------------------------------------------
# 3. Show the worst percentage errors
# ------------------------------------------------------------

worst_percentage = failure_df.sort_values(
    "Percentage_Error",
    ascending=False
).head(10)

print("\nTOP 10 WORST PREDICTIONS BY PERCENTAGE ERROR:")

display(
    worst_percentage[
        [
            "Actual_SalePrice",
            "Predicted_SalePrice",
            "Absolute_Error",
            "Percentage_Error"
        ]
    ]
)

# ------------------------------------------------------------
# 4. Summary of failure behavior
# ------------------------------------------------------------

print("\nFAILURE SUMMARY:")

print(
    f"Mean absolute error among test cases: "
    f"${failure_df['Absolute_Error'].mean():,.2f}"
)

print(
    f"Largest absolute error: "
    f"${failure_df['Absolute_Error'].max():,.2f}"
)

print(
    f"Largest percentage error: "
    f"{failure_df['Percentage_Error'].max():.2f}%"
)

print("\nFailure-example inspection completed.")

REAL FAILURE EXAMPLES — BASELINE MODEL

TOP 10 WORST PREDICTIONS BY ABSOLUTE ERROR:


,Actual_SalePrice,Predicted_SalePrice,Absolute_Error,Percentage_Error
691,755000,577451.865,177548.135,23.516309
898,611657,454077.790,157579.210,25.762676
1046,556581,423890.070,132690.930,23.840363
581,253293,365536.810,112243.810,44.313822
218,311500,209628.035,101871.965,32.703681
774,395000,308518.250,86481.750,21.894114
1243,465000,379215.855,85784.145,18.448203
231,403000,332986.595,70013.405,17.373053
1024,287000,352969.475,65969.475,22.985880
196,311872,247308.960,64563.040,20.701775



TOP 10 WORST PREDICTIONS BY PERCENTAGE ERROR:


,Actual_SalePrice,Predicted_SalePrice,Absolute_Error,Percentage_Error
30,40000,94992.600,54992.600,137.481500
916,35311,75990.500,40679.500,115.203478
1432,64500,110155.750,45655.750,70.784109
812,55993,94131.520,38138.520,68.113014
874,66500,105703.725,39203.725,58.952970
398,67000,104302.415,37302.415,55.675246
1279,68400,106059.500,37659.500,55.057749
581,253293,365536.810,112243.810,44.313822
589,79500,112619.270,33119.270,41.659459
628,135000,190953.835,55953.835,41.447285



FAILURE SUMMARY:
Mean absolute error among test cases: $17,386.11
Largest absolute error: $177,548.14
Largest percentage error: 137.48%

Failure-example inspection completed.


In [14]:
# ============================================================
# CELL 12 — FAILURE CASES WITH IMPORTANT FEATURES
# ============================================================

print("=" * 70)
print("FAILURE CASES — FEATURE-LEVEL INSPECTION")
print("=" * 70)

# Select the 10 largest absolute-error cases
worst_cases = failure_df.sort_values(
    "Absolute_Error",
    ascending=False
).head(10).copy()

# Add useful original features for interpretation
feature_columns = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "FullBath",
    "YearBuilt",
    "Neighborhood"
]

# Keep only columns that actually exist
available_features = [
    col for col in feature_columns
    if col in df.columns
]

# X_test contains the original feature values
inspection_df = X_test.loc[
    worst_cases.index,
    available_features
].copy()

# Add prediction results
inspection_df["Actual_SalePrice"] = worst_cases["Actual_SalePrice"]
inspection_df["Predicted_SalePrice"] = worst_cases["Predicted_SalePrice"]
inspection_df["Absolute_Error"] = worst_cases["Absolute_Error"]
inspection_df["Percentage_Error"] = worst_cases["Percentage_Error"]

# Reorder columns
inspection_df = inspection_df[
    available_features
    + [
        "Actual_SalePrice",
        "Predicted_SalePrice",
        "Absolute_Error",
        "Percentage_Error"
    ]
]

print("\nTOP 10 WORST CASES WITH FEATURES:")

display(inspection_df)

print("\nInterpretation guidance:")
print("""
- These are the test cases with the largest absolute prediction errors.
- Feature values are shown to identify possible patterns in model failures.
- A visible pattern is only an observation, not proof of causation.
- We should avoid claiming that any single feature caused the error
  unless additional analysis supports that conclusion.
""")

print("\nFeature-level failure inspection completed.")

FAILURE CASES — FEATURE-LEVEL INSPECTION

TOP 10 WORST CASES WITH FEATURES:


,OverallQual,GrLivArea,GarageCars,GarageArea,TotalBsmtSF,1stFlrSF,FullBath,YearBuilt,Neighborhood,Actual_SalePrice,Predicted_SalePrice,Absolute_Error,Percentage_Error
691,10,4316,3,832,2444,2444,3,1994,NoRidge,755000,577451.865,177548.135,23.516309
898,9,2364,3,820,2330,2364,2,2009,NridgHt,611657,454077.790,157579.210,25.762676
1046,9,2868,3,716,1992,1992,3,2005,StoneBr,556581,423890.070,132690.930,23.840363
581,8,2042,3,1390,2042,2042,2,2008,NridgHt,253293,365536.810,112243.810,44.313822
218,7,1954,2,431,798,1137,1,1939,Crawfor,311500,209628.035,101871.965,32.703681
774,8,1973,3,895,1935,1973,2,2006,NridgHt,395000,308518.250,86481.750,21.894114
1243,10,2076,3,850,2076,2076,2,2006,NridgHt,465000,379215.855,85784.145,18.448203
231,8,2794,3,810,1462,1490,2,1995,NoRidge,403000,332986.595,70013.405,17.373053
1024,8,2898,2,665,1565,2898,2,1976,Timber,287000,352969.475,65969.475,22.985880
196,7,1726,3,786,1726,1726,2,2007,Somerst,311872,247308.960,64563.040,20.701775



Interpretation guidance:

- These are the test cases with the largest absolute prediction errors.
- Feature values are shown to identify possible patterns in model failures.
- A visible pattern is only an observation, not proof of causation.
- We should avoid claiming that any single feature caused the error
  unless additional analysis supports that conclusion.


Feature-level failure inspection completed.


In [16]:
# ============================================================
# CELL 13 — FINAL VALIDATION & LEAKAGE SELF-CHECK
# ============================================================

print("=" * 70)
print("FINAL VALIDATION & LEAKAGE SELF-CHECK")
print("=" * 70)

checks = []

# ------------------------------------------------------------
# 1. TARGET LEAKAGE CHECK
# ------------------------------------------------------------

target_leakage = "SalePrice" in X.columns

print("\n1. TARGET LEAKAGE CHECK")
print(f"SalePrice present in model features: {target_leakage}")

checks.append(not target_leakage)


# ------------------------------------------------------------
# 2. IDENTIFIER CHECK
# ------------------------------------------------------------

id_leakage = "Id" in X.columns

print("\n2. IDENTIFIER CHECK")
print(f"Id present in model features: {id_leakage}")

checks.append(not id_leakage)


# ------------------------------------------------------------
# 3. DUPLICATE FEATURE CHECK
# ------------------------------------------------------------

duplicate_columns = X.columns[
    X.columns.duplicated()
].tolist()

print("\n3. DUPLICATE FEATURE CHECK")
print(f"Duplicate feature columns: {duplicate_columns}")

checks.append(len(duplicate_columns) == 0)


# ------------------------------------------------------------
# 4. TRAIN / TEST SIZE CHECK
# ------------------------------------------------------------

print("\n4. TRAIN / TEST SIZE CHECK")

print(f"Random split training rows: {len(X_train)}")
print(f"Random split test rows: {len(X_test)}")

print(f"Time-aware training rows: {len(X_time_train)}")
print(f"Time-aware test rows: {len(X_time_test)}")


# ------------------------------------------------------------
# 5. TIME-AWARE SPLIT AUDIT
# ------------------------------------------------------------

print("\n5. TIME-AWARE SPLIT AUDIT")

train_max_period = train_time["SalePeriod"].max()
test_min_period = test_time["SalePeriod"].min()

print(f"Latest training period: {train_max_period}")
print(f"Earliest test period: {test_min_period}")

if train_max_period < test_min_period:
    print("Chronological separation: PASS")
    chronological_separation = True
else:
    print(
        "Chronological separation: REVIEW — "
        "the cutoff occurs within the same sale month."
    )
    chronological_separation = False


# ------------------------------------------------------------
# 6. PREPROCESSING PIPELINE CHECK
# ------------------------------------------------------------

print("\n6. PREPROCESSING PIPELINE CHECK")

random_pipeline_ok = isinstance(baseline_model, Pipeline)
time_pipeline_ok = isinstance(time_model, Pipeline)

print(
    f"Random-split model uses Pipeline: "
    f"{random_pipeline_ok}"
)

print(
    f"Time-aware model uses Pipeline: "
    f"{time_pipeline_ok}"
)

print(
    "Preprocessing is fitted as part of the model pipeline, "
    "which helps prevent train/test preprocessing leakage."
)

pipeline_check = (
    random_pipeline_ok
    and time_pipeline_ok
)

checks.append(pipeline_check)


# ------------------------------------------------------------
# 7. VALIDATION RESULTS
# ------------------------------------------------------------

print("\n7. VALIDATION RESULTS")

print("\nRandom split:")
print(f"MAE : ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²  : {r2:.4f}")

print("\nTime-aware split:")
print(f"MAE : ${time_mae:,.2f}")
print(f"RMSE: ${time_rmse:,.2f}")
print(f"R²  : {time_r2:.4f}")


# ------------------------------------------------------------
# 8. FINAL AUDIT RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SELF-CHECK RESULT")
print("=" * 70)

if all(checks):
    print("PASS — No direct target leakage detected.")
    print("PASS — Identifier was removed before modeling.")
    print("PASS — No duplicate feature columns detected.")
    print("PASS — Preprocessing is contained inside model pipelines.")
else:
    print("REVIEW REQUIRED — One or more checks need attention.")


# ------------------------------------------------------------
# 9. METHODOLOGICAL NOTE
# ------------------------------------------------------------

print("\nMETHODOLOGICAL NOTE:")

print("""
- SalePrice is excluded from model features.
- Id is excluded because it is an identifier rather than a predictive input.
- Missing-value handling and categorical encoding are performed inside
  the sklearn pipelines.
- The time-aware split is ordered chronologically, but the 80% row cutoff
  falls within the same calendar month as the first test observations.
- Therefore, the time-aware result should be described as a later-observation
  holdout rather than a perfectly separated month-level forecast.
- Sale-related variables such as YrSold, MoSold, SaleType and SaleCondition
  require a clearly defined prediction point because their availability
  depends on when the prediction is intended to occur.
""")

print("\nFinal validation and leakage self-check completed.")

FINAL VALIDATION & LEAKAGE SELF-CHECK

1. TARGET LEAKAGE CHECK
SalePrice present in model features: False

2. IDENTIFIER CHECK
Id present in model features: False

3. DUPLICATE FEATURE CHECK
Duplicate feature columns: []

4. TRAIN / TEST SIZE CHECK
Random split training rows: 1168
Random split test rows: 292
Time-aware training rows: 1168
Time-aware test rows: 292

5. TIME-AWARE SPLIT AUDIT
Latest training period: 2009-07
Earliest test period: 2009-07
Chronological separation: REVIEW — the cutoff occurs within the same sale month.

6. PREPROCESSING PIPELINE CHECK
Random-split model uses Pipeline: True
Time-aware model uses Pipeline: True
Preprocessing is fitted as part of the model pipeline, which helps prevent train/test preprocessing leakage.

7. VALIDATION RESULTS

Random split:
MAE : $17,386.11
RMSE: $28,495.23
R²  : 0.8941

Time-aware split:
MAE : $17,181.08
RMSE: $25,948.76
R²  : 0.8835

FINAL SELF-CHECK RESULT
PASS — No direct target leakage detected.
PASS — Identifier was remov

## 14. Research Findings and Methodology Questions

### Finding 1 — Learned ranking performed substantially better than the hand-written baseline

The FlyRank starter research workflow reports that the random-forest ranking achieved approximately **0.740 Precision@50**, compared with approximately **0.240** for the hand-written baseline on the shipped starter data. The exact Random Forest value may vary by library version, so the main methodological point is the large difference between the two approaches rather than a specific third decimal.

**Methodology question:**
Was the improvement tested on a validation setup that closely represents how the ranking system would be used on genuinely future or unseen data? A stronger version would use a leakage-safe future-window evaluation before making broader performance claims.

---

### Finding 2 — Pre-decision features and leakage control are essential

The FlyRank methodology emphasizes using signals that are available before the decision being modeled. It also demonstrates a leakage problem where the label is derived from `trend_direction`, while `trend_pct` contains the underlying percentage change used to define that trend. Including `trend_pct` would therefore allow the model to see information closely related to the answer it is supposed to predict.

**Methodology question:**
For every feature, what was the exact information-availability point? A stronger validation process should explicitly document whether each feature would have been available before the prediction or ranking decision.

---

### Connection to my own model

In my housing-price experiment, I also performed a leakage and validation audit. `SalePrice` was excluded from the model features, `Id` was removed as an identifier, and preprocessing was placed inside the sklearn pipelines.

The audit also identified a limitation in my chronological holdout: the 80% row cutoff occurred within the same `SalePeriod` (`2009-07`) as the first test observations. Therefore, I should describe this as a **later-observation holdout**, rather than claiming that it is a perfectly separated month-level forecasting experiment.

This follows the same claim-discipline principle: report what the validation actually tested rather than making a stronger claim than the evidence supports.


## 15. Claim Rewrite — Evidence-Based Language

### Claim that would be too strong

> "The model accurately predicts house prices in the future."

### Evidence-based rewrite

> "On this dataset, the Random Forest achieved an RMSE of $25,948.76 and an MAE of $17,181.08 on the later-observation holdout. These results measure performance on the selected holdout and should not be interpreted as proof of real-world future forecasting performance."

### Another claim that would be too strong

> "The time-aware validation proves that the model will perform better in production."

### Evidence-based rewrite

> "The time-aware holdout produced a lower RMSE than the random split ($25,948.76 vs. $28,495.23), while R² was slightly lower (0.8835 vs. 0.8941). This is an observed difference between two validation designs, not evidence that the model will necessarily perform better in production."

### Leakage claim

> "The model has no leakage."

### Evidence-based rewrite

> "The audit found no direct target leakage: `SalePrice` was excluded from model features and `Id` was removed before training. However, sale-event variables such as `YrSold`, `MoSold`, `SaleType`, and `SaleCondition` require a clearly defined prediction point because their availability depends on when the prediction is intended to occur."

### Claim discipline principle

The results are reported as **observed and measured evidence** from this experiment. They are not presented as proof of causality, guaranteed future performance, or production-level impact.


In [18]:
# ============================================================
# CELL 16 — FINAL VALIDATION & LEAKAGE SELF-CHECK
# ============================================================

print("=" * 70)
print("FINAL VALIDATION & LEAKAGE SELF-CHECK")
print("=" * 70)

checks_passed = 0
checks_total = 6

# 1. Target leakage check
target_leakage = "SalePrice" in X.columns

if not target_leakage:
    print(" PASS — SalePrice is not included in model features.")
    checks_passed += 1
else:
    print(" FAIL — SalePrice is included in model features.")

# 2. Identifier check
id_present = "Id" in X.columns

if not id_present:
    print(" PASS — Id was removed before model training.")
    checks_passed += 1
else:
    print(" FAIL — Id is still present in model features.")

# 3. Duplicate feature columns
duplicate_columns = X.columns[X.columns.duplicated()].tolist()

if len(duplicate_columns) == 0:
    print(" PASS — No duplicate feature columns detected.")
    checks_passed += 1
else:
    print(" FAIL — Duplicate feature columns found:", duplicate_columns)

# 4. Preprocessing inside pipelines
baseline_is_pipeline = isinstance(baseline_model, Pipeline)
time_is_pipeline = isinstance(time_model, Pipeline)

if baseline_is_pipeline and time_is_pipeline:
    print(" PASS — Preprocessing is contained inside sklearn pipelines.")
    checks_passed += 1
else:
    print(" FAIL — Preprocessing pipeline structure needs review.")

# 5. Train/test size check
if len(X_train) == len(y_train) and len(X_test) == len(y_test):
    print(" PASS — Random split train/test sizes are consistent.")
    checks_passed += 1
else:
    print(" FAIL — Random split sizes are inconsistent.")

# 6. Time-aware chronological audit
latest_train_period = train_time["SalePeriod"].max()
earliest_test_period = test_time["SalePeriod"].min()

print("\nTime-aware validation audit:")
print("Latest training period :", latest_train_period)
print("Earliest testing period:", earliest_test_period)

if latest_train_period < earliest_test_period:
    print(" PASS — Clean chronological separation detected.")
    checks_passed += 1
else:
    print(" REVIEW — The cutoff occurs within the same sale month.")
    print("   This is a later-observation holdout, not a clean month-level split.")

# Final summary
print("\n" + "=" * 70)
print(f"SELF-CHECK RESULT: {checks_passed}/{checks_total} CORE CHECKS PASSED")
print("=" * 70)

print("\nValidation metrics:")
print(f"Random split   → MAE: ${mae:,.2f} | RMSE: ${rmse:,.2f} | R²: {r2:.4f}")
print(f"Time-aware     → MAE: ${time_mae:,.2f} | RMSE: ${time_rmse:,.2f} | R²: {time_r2:.4f}")

print("\nMethodological note:")
print("Sale-related variables such as YrSold, MoSold, SaleType, and")
print("SaleCondition require a clearly defined prediction point.")
print("They are not direct target leakage, but their availability depends")
print("on when the prediction is intended to be made.")

print("\nFinal conclusion:")
if target_leakage == False and id_present == False and len(duplicate_columns) == 0:
    print(" No direct target leakage was detected in the audited feature set.")
else:
    print(" Review the failed leakage checks before finalizing the notebook.")

FINAL VALIDATION & LEAKAGE SELF-CHECK
 PASS — SalePrice is not included in model features.
 PASS — Id was removed before model training.
 PASS — No duplicate feature columns detected.
 PASS — Preprocessing is contained inside sklearn pipelines.
 PASS — Random split train/test sizes are consistent.

Time-aware validation audit:
Latest training period : 2009-07
Earliest testing period: 2009-07
 REVIEW — The cutoff occurs within the same sale month.
   This is a later-observation holdout, not a clean month-level split.

SELF-CHECK RESULT: 5/6 CORE CHECKS PASSED

Validation metrics:
Random split   → MAE: $17,386.11 | RMSE: $28,495.23 | R²: 0.8941
Time-aware     → MAE: $17,181.08 | RMSE: $25,948.76 | R²: 0.8835

Methodological note:
Sale-related variables such as YrSold, MoSold, SaleType, and
SaleCondition require a clearly defined prediction point.
They are not direct target leakage, but their availability depends
on when the prediction is intended to be made.

Final conclusion:
 No direct

## 17. Deliverable Links

### Notebook / Repository

* GitHub Repository: **[Paste your GitHub repository URL here]**
* Executed Notebook: **[Paste your Google Colab / notebook URL here]**

### Required Video

* FlyRank Week 6 Recording: https://www.youtube.com/watch?v=0FPiCyYFX1k

### Supporting FlyRank Resources

* FlyRank Videos: https://www.youtube.com/@flyrank/videos
* Week 6 Q&A: https://internship.flyrank.ai/intern/events/ML-LIVE-06?tab=qa
* Week 6 Feedback: https://internship.flyrank.ai/intern/events/ML-LIVE-06?tab=feedback

### Completion Note

This notebook documents a validation and leakage audit of a Random Forest regression model on the Kaggle House Prices dataset. The analysis compares a random holdout with a later-observation chronological holdout, audits potential target leakage and feature availability, examines real prediction failures, and rewrites model claims using evidence-based language.

The validation results are reported as observed measurements from this experiment. No claims are made about guaranteed production performance, causality, or future forecasting beyond what the validation design supports.

### Final Methodological Limitation

The chronological holdout cutoff occurs within the same `SalePeriod` (`2009-07`). Therefore, the experiment is described as a **later-observation holdout**, rather than a perfectly separated month-level forecasting evaluation.

### Submission Checklist

* [x] Two research findings and methodology questions
* [x] Random-split baseline
* [x] Later-observation / time-aware validation
* [x] Before/after metric comparison
* [x] Leakage audit
* [x] Real failure examples
* [x] Feature-level failure inspection
* [x] Evidence-based claim rewrite
* [x] Final validation and leakage self-check
* [ ] GitHub repository URL added
* [ ] Executed notebook committed to repository
* [ ] Full Week 6 recording watched
* [x] Required video URL included in deliverables
